# 09 · Interoperability in practice — how another team merges its data with ours

**Question this notebook answers:** *what is JSON-LD actually good for?* The answer is shown with a small, concrete case.

**Scenario.** Two teams of a fictional company use our database to decide what to patch first.

| Who | What they have | Format | Real or invented |
|---|---|---|---|
| **ThreatGraph** (us) | KEV, NVD, ATT&CK, PKEV, linked | our API, `application/ld+json` | **real** (the loaded database) |
| **SOC team** | a log of security incidents, each one exploiting a CVE | their own JSON-LD, their own words (`soc:`) | **invented** for this demo |
| **Patch office** | a patch tracker: who owns the fix, planned date | a plain **CSV** with a column `cve` | **invented** for this demo |

**What we want to show**
1. The three sources are merged **without a conversion table and without our code**: they meet at the CVE identifier (`@id`).
2. The merged graph answers questions that **none of the three sources can answer alone**.
3. The answers are **correct**: each one is recomputed with pandas from the clean CSV files.
4. What goes wrong when the identifier is missing (negative control), and what is **not** solved by JSON-LD.

Steps: **A** our data → **B** the other teams' data → **C** merge → **D** questions → **E** checks → **F** negative control and provenance → **G** limits.

## A · Our side: ask the API for JSON-LD
The API is called in-process (no server to start), exactly as an external client would call it over HTTP.

In [1]:
import sys, os, json, random, datetime as dt
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "sources.yaml").exists())
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "webapp")]
os.environ["DJANGO_SETTINGS_MODULE"] = "threatgraph_site.settings"; os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
import django; django.setup()
from django.test import Client
from django.test.utils import setup_test_environment
from django.conf import settings
setup_test_environment(); settings.ALLOWED_HOSTS.append("127.0.0.1")
import pandas as pd
from IPython.display import display
from rdflib import Dataset, Graph, URIRef

web = Client(HTTP_HOST="127.0.0.1:8000")
JSON, LD = {"HTTP_ACCEPT": "application/json"}, {"HTTP_ACCEPT": "application/ld+json"}
OUT = ROOT / "data" / "exports" / "third_party"; OUT.mkdir(parents=True, exist_ok=True)
NVD = "https://nvd.nist.gov/vuln/detail/"          # the public IRI of a CVE: our @id AND their key to join

# all KEV CVEs that have at least one ATT&CK mapping (we need them for the demo)
cves, page = [], 1
while True:
    r = json.loads(web.get("/api/vulnerabilities/", {"has_attack": "true", "page_size": 200, "page": page}, **JSON).content)
    cves += [v["cve_id"] for v in r["results"]]
    if not r["next"]: break
    page += 1
print(len(cves), "KEV CVEs have at least one ATT&CK mapping")

419 KEV CVEs have at least one ATT&CK mapping


## B · The other teams' data (invented, deterministic)
A fixed random seed picks 40 CVEs. The **SOC** knows the first 24, the **patch office** knows 24 from the 11th on: **14 CVEs are in both**, 10 only in the SOC log, 10 only in the patch tracker.

In [2]:
pick = random.Random(42).sample(sorted(cves), 40)
soc_cves, patch_cves = pick[:24], pick[10:34]
print("in both:", len(set(soc_cves) & set(patch_cves)), "| only SOC:", len(set(soc_cves) - set(patch_cves)), "| only patch office:", len(set(patch_cves) - set(soc_cves)))

in both: 14 | only SOC: 10 | only patch office: 10


### B1 · SOC team: native JSON-LD with **its own vocabulary**
Their words (`soc:status`, `soc:system`) do not exist in our data. The only thing they share with us is the **identifier**: `soc:exploits` points to the NVD page of the CVE, i.e. the same IRI that is our `@id`.

In [3]:
SOC = "http://example.org/soc#"
kinds = ["web", "vpn", "mail", "file", "erp"]
soc_doc = {"@context": {"soc": SOC, "Incident": "soc:Incident", "status": "soc:status", "system": "soc:system",
                        "openedOn": {"@id": "soc:openedOn", "@type": "http://www.w3.org/2001/XMLSchema#date"},
                        "exploits": {"@id": "soc:exploits", "@type": "@id"}},
           "@graph": [{"@id": f"urn:example-soc:incident:INC-{i+1:03d}", "@type": "Incident",
                       "status": "open" if i % 3 else "closed",
                       "system": f"{kinds[i % 5]}-{10 + i:02d}",
                       "openedOn": str(dt.date(2025, 7, 1) + dt.timedelta(days=3 * i)),
                       "exploits": NVD + c} for i, c in enumerate(soc_cves)]}
(OUT / "soc_incidents.jsonld").write_text(json.dumps(soc_doc, indent=1))
print(json.dumps(soc_doc["@graph"][0], indent=1))

{
 "@id": "urn:example-soc:incident:INC-001",
 "@type": "Incident",
 "status": "closed",
 "system": "web-10",
 "openedOn": "2025-07-01",
 "exploits": "https://nvd.nist.gov/vuln/detail/CVE-2024-4761"
}


### B2 · Patch office: a **CSV**, not JSON-LD
They do not change their data. They keep the CSV and add **one rule**: *the `@id` of a CVE is the NVD URL*, plus a small context. That rule is the only "mapping" in the whole demo, and it is written once, at the edge.

In [4]:
owners = ["Web Platform", "Network", "Messaging", "Desktop"]
patch_df = pd.DataFrame({"cve": patch_cves,
                         "owner_team": [owners[j % 4] for j in range(len(patch_cves))],
                         "patched": ["yes" if j % 4 == 0 else "no" for j in range(len(patch_cves))],
                         "planned_date": [str(dt.date(2025, 8, 1) + dt.timedelta(days=5 * j)) for j in range(len(patch_cves))]})
patch_df.to_csv(OUT / "patch_tracker.csv", index=False)
display(patch_df.head(4))

PT = "http://example.org/patching#"
patch_doc = {"@context": {"@vocab": PT, "forCve": {"@type": "@id"}},      # every column becomes a property of pt:
             "@graph": [{"@id": f"urn:example-patch:item:{j+1:03d}", "@type": "Item",
                         "forCve": NVD + r.cve, "owner_team": r.owner_team, "patched": r.patched,
                         "planned_date": r.planned_date} for j, r in enumerate(patch_df.itertuples())]}
(OUT / "patch_tracker.jsonld").write_text(json.dumps(patch_doc, indent=1))
print(json.dumps(patch_doc["@graph"][0], indent=1))

,cve,owner_team,patched,planned_date
0,CVE-2024-57727,Web Platform,yes,2025-08-01
1,CVE-2023-48788,Network,no,2025-08-06
2,CVE-2017-11292,Messaging,no,2025-08-11
3,CVE-2024-21413,Desktop,no,2025-08-16


{
 "@id": "urn:example-patch:item:001",
 "@type": "Item",
 "forCve": "https://nvd.nist.gov/vuln/detail/CVE-2024-57727",
 "owner_team": "Web Platform",
 "patched": "yes",
 "planned_date": "2025-08-01"
}


## C · Merge
We fetch **our** JSON-LD only for the CVEs that the two teams mention (as a real client would), then load the three documents into one RDF *dataset*.
Each document goes into its **own named graph**, so after the merge we still know **who said what** (see section F).

In [5]:
needed = sorted(set(soc_cves) | set(patch_cves))
ours = [json.loads(web.get(f"/api/vulnerabilities/{c}/", **LD).content) for c in needed]
(OUT / "threatgraph_vulnerabilities.jsonld").write_text(json.dumps(ours))
print(len(ours), "JSON-LD documents from our API;", sum(len(json.dumps(d)) for d in ours) // 1024, "KB")

ds = Dataset(default_union=True)                       # queries see the union of all graphs
G = {"threatgraph": "urn:graph:threatgraph", "soc": "urn:graph:soc", "patch": "urn:graph:patch"}
for d in ours:
    ds.graph(URIRef(G["threatgraph"])).parse(data=json.dumps(d), format="json-ld")
ds.graph(URIRef(G["soc"])).parse(data=json.dumps(soc_doc), format="json-ld")
ds.graph(URIRef(G["patch"])).parse(data=json.dumps(patch_doc), format="json-ld")
sizes = {k: len(ds.graph(URIRef(u))) for k, u in G.items()}
print("triples per graph:", sizes, "| total:", sum(sizes.values()))

34 JSON-LD documents from our API; 293 KB
triples per graph: {'threatgraph': 2393, 'soc': 120, 'patch': 120} | total: 2633


## D · Questions that only the merge can answer

In [6]:
P = ("PREFIX tg: <https://example.org/threatgraph/vocab#> PREFIX dct: <http://purl.org/dc/terms/> "
     f"PREFIX soc: <{SOC}> PREFIX pt: <{PT}> ")
def ask(q):
    res = ds.query(P + q)
    return pd.DataFrame([[(x.toPython() if x is not None else None) for x in row] for row in res], columns=[str(v) for v in res.vars])

**Q1 · Open incidents: how dangerous is the exploited vulnerability, and which ATT&CK tactic does it serve?**
`soc:` (status, incident) + `tg:` (product, ransomware use, CVSS) + ATT&CK tactic (through our PKEV mapping). Three kinds of knowledge from two organisations.

In [7]:
q1 = ask('''
SELECT ?incident ?cve ?product ?ransomware ?score (GROUP_CONCAT(DISTINCT ?tactic; separator=", ") AS ?tactics) WHERE {
  ?i a soc:Incident ; soc:status "open" ; soc:exploits ?v .
  ?v dct:identifier ?cve ; tg:product ?product ; tg:ransomwareUse ?ransomware ;
     tg:nvdRecord/tg:cvssV31/tg:baseScore ?score .
  OPTIONAL { ?v tg:hasMapping/tg:technique/tg:tactic/tg:shortname ?tactic }
  BIND(REPLACE(STR(?i), "^.*:", "") AS ?incident)
} GROUP BY ?incident ?cve ?product ?ransomware ?score ORDER BY DESC(?score) ?incident''')
print(len(q1), "open incidents"); display(q1.head(8))

16 open incidents


,incident,cve,product,ransomware,score,tactics
0,INC-002,CVE-2018-4939,ColdFusion,unknown,9.8,"initial-access, execution, persistence"
1,INC-003,CVE-2011-2462,Acrobat and Reader,unknown,9.8,execution
2,INC-008,CVE-2019-19781,"Application Delivery Controller (ADC), Gateway...",known,9.8,"execution, discovery, initial-access, persistence"
3,INC-012,CVE-2023-48788,FortiClient EMS,known,9.8,"initial-access, execution, command-and-control"
4,INC-014,CVE-2024-21413,Office Outlook,unknown,9.8,"initial-access, execution"
5,INC-018,CVE-2017-5638,Struts,known,9.8,"initial-access, execution, collection"
6,INC-021,CVE-2023-36845,Junos OS,unknown,9.8,"initial-access, execution"
7,INC-005,CVE-2021-37975,Chromium V8,unknown,8.8,execution


**Q2 · What should be patched first?** Open incident **and** patch not done **and** ransomware use known. Three sources, one query: the SOC says *incident*, the patch office says *owner and date*, we say *ransomware and score*.

In [8]:
q2 = ask('''
SELECT ?cve ?product ?owner ?planned ?score WHERE {
  ?i a soc:Incident ; soc:status "open" ; soc:exploits ?v .
  ?p pt:forCve ?v ; pt:owner_team ?owner ; pt:patched "no" ; pt:planned_date ?planned .
  ?v dct:identifier ?cve ; tg:product ?product ; tg:ransomwareUse "known" ; tg:nvdRecord/tg:cvssV31/tg:baseScore ?score .
} ORDER BY ?planned''')
print(len(q2), "items to patch first"); display(q2)

3 items to patch first


,cve,product,owner,planned,score
0,CVE-2023-48788,FortiClient EMS,Network,2025-08-06,9.8
1,CVE-2017-5638,Struts,Desktop,2025-09-05,9.8
2,CVE-2021-27065,Exchange Server,Network,2025-09-15,7.8


**Q3 · Which tactics do the open incidents belong to?** (a view for the SOC manager)

In [9]:
q3 = (q1.assign(tactic=q1.tactics.str.split(", ")).explode("tactic").dropna(subset=["tactic"])
        .groupby("tactic").incident.nunique().sort_values(ascending=False).rename("open incidents").to_frame())
display(q3)

,open incidents
tactic,
execution,13
initial-access,10
persistence,4
command-and-control,3
impact,2
privilege-escalation,2
defense-evasion,2
credential-access,1
collection,1


## E · Checks: the same answers recomputed **without RDF**
For each question, the expected result is recomputed with **pandas from the clean CSV files** (not from the database, not from the API, not from JSON-LD). If the identifier-based merge works, the two methods agree.

In [10]:
CL = ROOT / "data" / "clean"
kev = pd.read_csv(CL / "kev_vulnerability.csv"); nvd = pd.read_csv(CL / "nvd_cve.csv")
pm = pd.read_csv(CL / "pkev_mapping.csv"); tt = pd.read_csv(CL / "attack_technique_tactic.csv"); tac = pd.read_csv(CL / "attack_tactic.csv")
soc_df = pd.DataFrame([{"incident": g["@id"].split(":")[-1], "cve": g["exploits"].rsplit("/", 1)[1], "status": g["status"]} for g in soc_doc["@graph"]])
open_df = soc_df[soc_df.status == "open"]

tactics = (pm[["cve_id", "technique_id"]].drop_duplicates().merge(tt, on="technique_id").merge(tac[["tactic_id", "shortname"]], on="tactic_id")
             .groupby("cve_id").shortname.apply(lambda s: ", ".join(sorted(set(s)))))
exp1 = (open_df.merge(kev[["cve_id", "product", "ransomware_use"]], left_on="cve", right_on="cve_id")
               .merge(nvd[["cve_id", "cvss_v31_score"]], on="cve_id"))
exp1["tactics"] = exp1.cve.map(tactics).fillna("")
exp1 = {(r.incident, r.cve, r.product, r.ransomware_use, float(r.cvss_v31_score), r.tactics) for r in exp1.itertuples()}
got1 = {(r.incident, r.cve, r.product, r.ransomware, float(r.score), ", ".join(sorted(set(r.tactics.split(", ")))) if r.tactics else "") for r in q1.itertuples()}

exp2 = (open_df.merge(patch_df[patch_df.patched == "no"], on="cve").merge(kev[["cve_id", "product", "ransomware_use"]], left_on="cve", right_on="cve_id")
               .query("ransomware_use == 'known'"))
exp2 = {(r.cve, r.owner_team, r.planned_date) for r in exp2.itertuples()}
got2 = {(r.cve, r.owner, str(r.planned)) for r in q2.itertuples()}

# Q3 counts incidents per tactic (one incident may have several tactics)
exp3 = {t: len({x[0] for x in exp1 if t in x[5].split(", ")}) for t in {t for x in exp1 for t in x[5].split(", ") if t}}
checks = [("Q1 same rows (incident, CVE, product, ransomware, CVSS, tactics) as pandas", got1 == exp1, f"{len(got1)} vs {len(exp1)}"),
          ("Q1 every open incident found (none lost in the merge)", len(q1) == len(open_df), f"{len(q1)} vs {len(open_df)}"),
          ("Q2 same items as pandas", got2 == exp2, f"{len(got2)} vs {len(exp2)}"),
          ("Q2 uses all three sources (no row from a two-way join slips in)", all(c in set(soc_cves) & set(patch_cves) for c, *_ in got2), f"{len(got2)} items"),
          ("Q3 incidents per tactic equal pandas", dict(q3["open incidents"]) == dict(exp3), f"{len(q3)} tactics")]
for name, ok, info in checks: print("✓" if ok else "✗", name, "·", info)
assert all(ok for _, ok, _ in checks)

✓ Q1 same rows (incident, CVE, product, ransomware, CVSS, tactics) as pandas · 16 vs 16
✓ Q1 every open incident found (none lost in the merge) · 16 vs 16
✓ Q2 same items as pandas · 3 vs 3
✓ Q2 uses all three sources (no row from a two-way join slips in) · 3 items
✓ Q3 incidents per tactic equal pandas · 11 tactics


## F · What if the identifier is missing? (negative control) and: who said what?

**F1.** A third team writes the CVE as plain text (`"CVE-2021-44228"`) instead of a link. The data is there, but it **does not meet** ours: the graphs stay apart. This is the whole point of the `@id`: it is not decoration.

In [11]:
bad = {"@context": {"soc": SOC}, "@id": "urn:example-soc:incident:INC-999", "@type": "soc:Incident",
       "soc:status": "open", "soc:exploits": "CVE-2021-44228"}                      # a string, not an IRI
good = {"@context": {"soc": SOC, "soc:exploits": {"@type": "@id"}}, "@id": "urn:example-soc:incident:INC-998", "@type": "soc:Incident",
        "soc:status": "open", "soc:exploits": NVD + "CVE-2021-44228"}               # the IRI of the CVE
log4shell = json.loads(web.get("/api/vulnerabilities/CVE-2021-44228/", **LD).content)      # one real document of ours
def joined(doc):
    d = Dataset(default_union=True)
    d.graph(URIRef(G["threatgraph"])).parse(data=json.dumps(log4shell), format="json-ld")
    d.graph(URIRef("urn:graph:third")).parse(data=json.dumps(doc), format="json-ld")
    return len(list(d.query(P + "SELECT ?i ?p WHERE { ?i soc:exploits ?v . ?v tg:product ?p }")))
print("plain text  -> rows joined:", joined(bad))
print("@id (IRI)   -> rows joined:", joined(good))
assert joined(bad) == 0 and joined(good) == 1

plain text  -> rows joined: 0
@id (IRI)   -> rows joined: 1


**F2.** After the merge we can still ask **which source** contributed what, thanks to the named graphs.

In [12]:
cve = NVD + soc_cves[0] if soc_cves[0] in patch_cves else NVD + sorted(set(soc_cves) & set(patch_cves))[0]
res = ds.query(P + f'''SELECT ?g (COUNT(*) AS ?triples) WHERE {{ GRAPH ?g {{ ?s ?p ?o . FILTER(?s = <{cve}> || ?o = <{cve}>) }} }} GROUP BY ?g ORDER BY ?g''')
print("triples that mention", cve.rsplit("/", 1)[1])
display(pd.DataFrame([[str(r[0]), int(r[1])] for r in res], columns=["graph (who said it)", "triples"]))

triples that mention CVE-2012-0754


,graph (who said it),triples
0,urn:graph:patch,1
1,urn:graph:soc,1
2,urn:graph:threatgraph,18


## G · What this shows, and what it does not

**Shown**
* Three independent sources, one of them a **CSV that was never JSON-LD**, were combined by sharing only the **CVE identifier**: no mapping table, no code of ours (`rdflib` is a generic library).
* The merged graph answers questions that need all three (Q2), and the answers match a recomputation with pandas on the clean tables.
* The identifier matters: with a plain-text CVE the data **silently does not merge** (F1).
* The merge keeps provenance if each file goes to its own named graph (F2).

**Not shown, and honest limits** [INFERENCE]
* **The other teams' data are invented.** The demo proves the mechanism, not that a real SOC would find our data useful.
* **Identifier convention.** The join works only if every team writes the CVE as `https://nvd.nist.gov/vuln/detail/CVE-…`. A different but equivalent URL (for example a cve.org page) would **not** merge, and nothing would warn. A real project needs an agreed convention or `owl:sameAs` links.
* **Meaning is not aligned.** `soc:status "open"` and our terms live side by side; nobody states that they are related. The merge joins *things*, it does not reconcile *vocabularies*.
* **`tg:` is a placeholder namespace** (`https://example.org/threatgraph/vocab#`): a human or program outside our project cannot look up what `tg:ransomwareUse` means. This limits semantic interoperability (FAIR I2).
* **No conflict handling.** If two sources gave different values for the same property of the same node, the graph would simply hold both.
* The queries run on a small subset (40 CVEs) in memory, which is no performance argument for the full data.

**One-sentence version for the exam:** *"Our JSON-LD gives every CVE, technique and weakness a public identifier. A team that uses the same identifier can add its own data to ours and ask joint questions without a conversion table, as notebook 09 shows with a SOC log and a patch tracker."*